In [1]:
# Core imports
import os
import sys
import argparse
import shutil
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# Optional imports with fallbacks
try:
    import seaborn as sns
    print("✅ Seaborn available for enhanced plotting")
    HAS_SEABORN = True
except ImportError:
    print("⚠️ Seaborn not found - installing...")
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn"])
        import seaborn as sns
        print("✅ Seaborn installed and imported successfully")
        HAS_SEABORN = True
    except Exception as e:
        print(f"⚠️ Could not install seaborn: {e}")
        print("📊 Will use matplotlib for basic plotting")
        HAS_SEABORN = False

# Hyperparameter optimization
try:
    import optuna
    print("✅ Optuna available for optimization")
except ImportError:
    print("⚠️ Installing Optuna for hyperparameter optimization...")
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])
        import optuna
        print("✅ Optuna installed successfully")
    except Exception as e:
        print(f"❌ Failed to install Optuna: {e}")
        print("💡 Please install manually: pip install optuna")

# Plotting
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    print("✅ Plotly available for interactive plots")
except ImportError:
    print("⚠️ Installing Plotly for interactive visualization...")
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly"])
        import plotly.graph_objects as go
        import plotly.express as px
        from plotly.subplots import make_subplots
        print("✅ Plotly installed successfully")
    except Exception as e:
        print(f"⚠️ Could not install plotly: {e}")
        print("📊 Will use matplotlib for basic plotting")

# Import existing CoFT functionality - INHERIT, DON'T REWRITE
try:
    from main import execute_training_mode
    print("✅ Imported existing training function from main.py")
except ImportError:
    print("❌ Cannot import from main.py - ensure you're in CoFT project directory")
    print("💡 Current directory:", os.getcwd())
    print("💡 Available files:", [f for f in os.listdir('.') if f.endswith('.py')])
    print("💡 Try: cd /path/to/CoFT/project")

# Environment check
try:
    import torch
    print(f"✅ PyTorch available: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"🚀 CUDA available: {torch.cuda.get_device_name()}")
    else:
        print("⚙️ CUDA not available, will use CPU")
except ImportError:
    print("⚠️ PyTorch not found - ensure you're in the CoFT environment")

# Configure environment
plt.style.use('default')
if HAS_SEABORN:
    sns.set_palette("husl")

print("\\n🚀 CoFT Hyperparameter Tuning Environment Ready")
print(f"📍 Working Directory: {os.getcwd()}")
print(f"📊 Python Version: {sys.version.split()[0]}")
print(f"🐍 Python Executable: {sys.executable}")

# Environment verification
if 'CoFT' in sys.executable or 'coft' in sys.executable.lower():
    print("✅ Running in CoFT environment")
else:
    print("⚠️ Not running in CoFT environment")
    print("💡 To fix: activate CoFT environment or change Jupyter kernel")


✅ Seaborn available for enhanced plotting
✅ Optuna available for optimization


c:\Users\Huy\anaconda3\envs\CoFT\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Plotly available for interactive plots
✅ Imported existing training function from main.py
✅ PyTorch available: 2.4.1
🚀 CUDA available: NVIDIA GeForce RTX 4060
\n🚀 CoFT Hyperparameter Tuning Environment Ready
📍 Working Directory: d:\Project\CoFT
📊 Python Version: 3.8.20
🐍 Python Executable: c:\Users\Huy\anaconda3\envs\CoFT\python.exe
✅ Running in CoFT environment


In [2]:
# =============================================================================
# CONFIGURATION - Modify these settings for your optimization
# =============================================================================

class OptimizationConfig:
    """Configuration for hyperparameter optimization."""
    
    def __init__(self):
        # Experiment settings
        self.experiment_description = 'hyperparameter_tuning'
        self.run_description = f'optuna_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
        self.seed = 42
        
        # Dataset selection
        self.selected_dataset = 'HAR'  # Change as needed: 'HAR', 'sleep', 'Epilepsy', 'pFD'
        self.data_path = 'data/'
        
        # Optimization mode
        self.optimization_mode = 'diagnostic'  # 'diagnostic', 'quick', 'full'
        self.n_trials = {'diagnostic': 5, 'quick': 20, 'full': 100}
        
        # System settings
        self.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        self.logs_save_dir = 'tuning_experiments'
        self.home_path = os.getcwd()
        
        # Results directory
        self.results_dir = f"hyperparameter_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        os.makedirs(self.results_dir, exist_ok=True)
        
        # Training mode for optimization (fixed to ft_1p for parameter testing)
        self.training_mode = 'ft_1p'
        
        # Debugging and safety options
        self.enable_debugging = True     # Enable detailed logging and checks
        self.enable_gradient_clipping = True  # Prevent NaN losses
        self.max_trial_duration = 900    # 15 minutes timeout per trial
        
        # Parameter search spaces (based on proven optimal ranges)
        self.search_spaces = {
            'diagnostic': {
                'lambda_cotraining': [0.001, 0.005, 0.01],
                'lambda_consistency': [0.1],  # Fixed based on analysis
                'ensemble_method': ['temporal_only', 'simple_average']
            },
            'quick': {
                'lambda_cotraining': [0.0005, 0.001, 0.002, 0.005, 0.01],
                'lambda_consistency': [0.1, 0.2],
                'ensemble_method': ['temporal_only', 'simple_average']
            },
            'full': {
                'lambda_cotraining': (0.0001, 0.05, 'log'),  # Log-uniform distribution
                'lambda_consistency': (0.05, 0.5, 'uniform'),
                'ensemble_method': ['temporal_only', 'simple_average']
            }
        }

# Initialize configuration
config = OptimizationConfig()

print("🎯 Optimization Configuration")
print(f"   Dataset: {config.selected_dataset}")
print(f"   Mode: {config.optimization_mode}")
print(f"   Trials: {config.n_trials[config.optimization_mode]}")
print(f"   Device: {config.device}")
print(f"   Results: {config.results_dir}")
print(f"   Search Space: {config.search_spaces[config.optimization_mode]}")


🎯 Optimization Configuration
   Dataset: HAR
   Mode: diagnostic
   Trials: 5
   Device: cuda:0
   Results: hyperparameter_results_20250622_193754
   Search Space: {'lambda_cotraining': [0.001, 0.005, 0.01], 'lambda_consistency': [0.1], 'ensemble_method': ['temporal_only', 'simple_average']}


In [3]:
# =============================================================================
# PARAMETER UPDATE UTILITIES - Inherited from optimize_coft.sh logic
# =============================================================================

import re  # Add missing import for regex operations

class CoFTParameterManager:
    """
    Manages CoFT parameter updates for hyperparameter optimization.
    Inherits logic from optimize_coft.sh bash script.
    """
    
    def __init__(self):
        self.coft_loss_file = 'models/coft_loss.py'
        self.trainer_file = 'trainer/trainer_coft.py'
        self.backup_suffix = '.backup'
        
    def backup_files(self):
        """Create backup copies of parameter files."""
        for file_path in [self.coft_loss_file, self.trainer_file]:
            if os.path.exists(file_path):
                shutil.copy2(file_path, file_path + self.backup_suffix)
                print(f"   📝 Backed up: {file_path}")
    
    def restore_files(self):
        """Restore files from backup."""
        for file_path in [self.coft_loss_file, self.trainer_file]:
            backup_path = file_path + self.backup_suffix
            if os.path.exists(backup_path):
                shutil.copy2(backup_path, file_path)
                os.remove(backup_path)
                print(f"   🔄 Restored: {file_path}")
    
    def update_coft_loss_params(self, lambda_cotraining: float, lambda_consistency: float):
        """
        Update CoFT loss parameters in models/coft_loss.py.
        Inherited from bash script update_coft_loss_params() function.
        """
        if not os.path.exists(self.coft_loss_file):
            raise FileNotFoundError(f"CoFT loss file not found: {self.coft_loss_file}")
        
        # Read current file
        with open(self.coft_loss_file, 'r') as f:
            content = f.read()
        
        # Update parameters using regex (similar to sed in bash script)
        
        # Update lambda_cotraining
        content = re.sub(
            r'self\.lambda_cotraining = [0-9]*\.?[0-9]*',
            f'self.lambda_cotraining = {lambda_cotraining}',
            content
        )
        
        # Update lambda_consistency  
        content = re.sub(
            r'self\.lambda_consistency = [0-9]*\.?[0-9]*',
            f'self.lambda_consistency = {lambda_consistency}',
            content
        )
        
        # Write updated content
        with open(self.coft_loss_file, 'w') as f:
            f.write(content)
        
        print(f"   ✅ Updated λ_cotraining={lambda_cotraining}, λ_consistency={lambda_consistency}")
    
    def update_ensemble_method(self, method: str):
        """
        Update ensemble method in trainer/trainer_coft.py.
        Now supports both temporal_only and simple_average modes.
        """
        if not os.path.exists(self.trainer_file):
            print(f"   ⚠️ Trainer file not found: {self.trainer_file} - skipping ensemble update")
            return
        
        # Read current file
        with open(self.trainer_file, 'r') as f:
            content = f.read()
        
        if method == 'temporal_only':
            # Replace ensemble logic with temporal-only
            content = re.sub(
                r'final_predictions = \(predictions \+ freq_predictions\) / 2  # SIMPLE_AVERAGE',
                'final_predictions = predictions  # TEMPORAL_ONLY_MODE',
                content
            )
            # Also handle case where ensemble_predictions exists
            content = re.sub(
                r'ensemble_predictions = ensemble_module\(predictions, freq_predictions\)\s*\n\s*final_predictions = \(predictions \+ freq_predictions\) / 2  # SIMPLE_AVERAGE',
                'final_predictions = predictions  # TEMPORAL_ONLY_MODE',
                content
            )
        elif method == 'simple_average':
            # Restore ensemble logic
            content = re.sub(
                r'final_predictions = predictions  # TEMPORAL_ONLY_MODE',
                'ensemble_predictions = ensemble_module(predictions, freq_predictions)\n                     final_predictions = (predictions + freq_predictions) / 2  # SIMPLE_AVERAGE',
                content
            )
        
        # Write updated content
        with open(self.trainer_file, 'w') as f:
            f.write(content)
        
        print(f"   ✅ Updated ensemble method: {method}")
    
    def verify_parameters(self, lambda_cotraining: float, lambda_consistency: float, 
                         ensemble_method: str) -> int:
        """
        Enhanced parameter verification with detailed checking.
        """
        verification_score = 0
        issues = []
        
        # Check lambda_cotraining in CoFT loss file
        if os.path.exists(self.coft_loss_file):
            with open(self.coft_loss_file, 'r') as f:
                content = f.read()
                # More flexible regex matching
                import re
                lambda_ct_pattern = rf'self\.lambda_cotraining\s*=\s*{lambda_cotraining}'
                if re.search(lambda_ct_pattern, content):
                    verification_score += 1
                    print(f"   ✅ lambda_cotraining = {lambda_cotraining} verified")
                else:
                    issues.append(f"lambda_cotraining = {lambda_cotraining} not found")
        
        # Check lambda_consistency
        if os.path.exists(self.coft_loss_file):
            with open(self.coft_loss_file, 'r') as f:
                content = f.read()
                lambda_cs_pattern = rf'self\.lambda_consistency\s*=\s*{lambda_consistency}'
                if re.search(lambda_cs_pattern, content):
                    verification_score += 1
                    print(f"   ✅ lambda_consistency = {lambda_consistency} verified")
                else:
                    issues.append(f"lambda_consistency = {lambda_consistency} not found")
        
        # Check ensemble method in trainer file
        if os.path.exists(self.trainer_file):
            with open(self.trainer_file, 'r') as f:
                content = f.read()
                if ensemble_method == 'temporal_only':
                    if 'final_predictions = predictions  # TEMPORAL_ONLY_MODE' in content:
                        verification_score += 1
                        print(f"   ✅ Ensemble method: {ensemble_method} verified")
                    else:
                        issues.append(f"temporal_only mode not properly set")
                elif ensemble_method == 'simple_average':
                    if 'final_predictions = (predictions + freq_predictions) / 2  # SIMPLE_AVERAGE' in content:
                        verification_score += 1
                        print(f"   ✅ Ensemble method: {ensemble_method} verified")
                    else:
                        issues.append(f"simple_average mode not properly set")
        
        if issues:
            print(f"   ⚠️ Verification issues: {issues}")
        
        return verification_score

# Initialize parameter manager
param_manager = CoFTParameterManager()
print("🔧 Parameter Manager initialized")
print("   ✅ Ready to update CoFT loss parameters")
print("   ✅ Ready to update ensemble methods")
print("   ✅ Backup and restore capabilities enabled")


🔧 Parameter Manager initialized
   ✅ Ready to update CoFT loss parameters
   ✅ Ready to update ensemble methods
   ✅ Backup and restore capabilities enabled


In [4]:
# =============================================================================
# HYPERPARAMETER OPTIMIZATION ENGINE - Using Optuna
# =============================================================================

class CoFTHyperparameterOptimizer:
    """
    Intelligent hyperparameter optimization using Optuna.
    Replaces manual parameter loops from bash script with smart search.
    """
    
    def __init__(self, config: OptimizationConfig, param_manager: CoFTParameterManager):
        self.config = config
        self.param_manager = param_manager
        self.results = []
        self.best_result = None
        
        # Create Optuna study
        self.study = optuna.create_study(
            direction='maximize',  # Maximize test accuracy
            study_name=f"coft_optimization_{config.selected_dataset}_{config.optimization_mode}",
            storage=f"sqlite:///{config.results_dir}/optuna_study.db",
            load_if_exists=True
        )
        
    def create_trial_params(self, trial: optuna.Trial) -> Dict:
        """Create hyperparameters for a trial based on search space."""
        search_space = self.config.search_spaces[self.config.optimization_mode]
        params = {}
        
        # Lambda cotraining
        if isinstance(search_space['lambda_cotraining'], list):
            params['lambda_cotraining'] = trial.suggest_categorical(
                'lambda_cotraining', search_space['lambda_cotraining']
            )
        else:
            low, high, distribution = search_space['lambda_cotraining']
            if distribution == 'log':
                params['lambda_cotraining'] = trial.suggest_loguniform(
                    'lambda_cotraining', low, high
                )
            else:
                params['lambda_cotraining'] = trial.suggest_uniform(
                    'lambda_cotraining', low, high
                )
        
        # Lambda consistency
        if isinstance(search_space['lambda_consistency'], list):
            params['lambda_consistency'] = trial.suggest_categorical(
                'lambda_consistency', search_space['lambda_consistency']
            )
        else:
            low, high, distribution = search_space['lambda_consistency']
            params['lambda_consistency'] = trial.suggest_uniform(
                'lambda_consistency', low, high
            )
        
        # Ensemble method
        params['ensemble_method'] = trial.suggest_categorical(
            'ensemble_method', search_space['ensemble_method']
        )
        
        return params
    
    def objective(self, trial: optuna.Trial) -> float:
        """
        Objective function for Optuna optimization.
        Uses existing execute_training_mode function - INHERITS LOGIC.
        """
        trial_start_time = datetime.now()
        
        # Get trial parameters
        params = self.create_trial_params(trial)
        
        print(f"\\n🔬 Trial {trial.number + 1}: Testing parameters")
        print(f"   λ_cotraining: {params['lambda_cotraining']}")
        print(f"   λ_consistency: {params['lambda_consistency']}")
        print(f"   ensemble: {params['ensemble_method']}")
        
        try:
            # Backup original files
            self.param_manager.backup_files()
            
            # Update parameters
            self.param_manager.update_coft_loss_params(
                params['lambda_cotraining'],
                params['lambda_consistency']
            )
            self.param_manager.update_ensemble_method(params['ensemble_method'])
            
            # Enhanced parameter verification
            verification_score = self.param_manager.verify_parameters(
                params['lambda_cotraining'],
                params['lambda_consistency'],
                params['ensemble_method']
            )
            
            if verification_score < 2:  # At least 2/3 parameters verified
                print(f"   ❌ Parameter verification failed: {verification_score}/3")
                print(f"   💡 Check if parameter update patterns match actual code")
                raise optuna.TrialPruned()
            else:
                print(f"   ✅ Parameter verification passed: {verification_score}/3")
            
            # Create arguments object for existing training function
            args = argparse.Namespace(
                experiment_description=self.config.experiment_description,
                run_description=f"{self.config.run_description}_trial_{trial.number}",
                seed=self.config.seed,
                training_mode=self.config.training_mode,
                selected_dataset=self.config.selected_dataset,
                data_path=self.config.data_path,
                logs_save_dir=self.config.logs_save_dir,
                device=self.config.device,
                home_path=self.config.home_path,
                enable_coft=True  # Always enable CoFT for parameter optimization
            )
            
            # For ft_1p mode, ensure base model is available for this trial
            if self.config.training_mode == 'ft_1p':
                self.setup_base_model_for_trial(trial.number)
            
            # Execute training using existing function - INHERIT LOGIC
            print(f"   ⏳ Running training with {self.config.training_mode} mode...")
            
            success = execute_training_mode(args, self.config.training_mode, trial_start_time)
            
            if not success:
                print(f"   ❌ Training failed for trial {trial.number}")
                raise optuna.TrialPruned()
            
            # Extract test accuracy from logs
            log_dir = os.path.join(
                self.config.logs_save_dir,
                self.config.experiment_description,
                f"{self.config.run_description}_trial_{trial.number}",
                f"{self.config.training_mode}_seed_{self.config.seed}"
            )
            
            test_accuracy = self.extract_test_accuracy(log_dir)
            
            if test_accuracy is None:
                print(f"   ⚠️ Could not extract test accuracy for trial {trial.number}")
                raise optuna.TrialPruned()
            
            # Store result
            trial_duration = (datetime.now() - trial_start_time).total_seconds()
            result = {
                'trial': trial.number,
                'lambda_cotraining': params['lambda_cotraining'],
                'lambda_consistency': params['lambda_consistency'], 
                'ensemble_method': params['ensemble_method'],
                'test_accuracy': test_accuracy,
                'duration_seconds': trial_duration,
                'verification_score': verification_score,
                'timestamp': datetime.now().isoformat()
            }
            
            self.results.append(result)
            self.save_results()
            
            print(f"   ✅ Trial {trial.number + 1} completed: {test_accuracy:.4f}% accuracy")
            
            # Update best result if improved
            if self.best_result is None or test_accuracy > self.best_result['test_accuracy']:
                self.best_result = result.copy()
                print(f"   🏆 NEW BEST: {test_accuracy:.4f}% accuracy!")
            
            return test_accuracy
            
        except Exception as e:
            print(f"   ❌ Trial {trial.number} failed: {str(e)}")
            raise optuna.TrialPruned()
            
        finally:
            # Always restore original files
            self.param_manager.restore_files()
    
    def extract_test_accuracy(self, log_dir: str) -> Optional[float]:
        """Extract test accuracy from training logs."""
        import glob
        import re
        
        # Find log files
        log_pattern = os.path.join(log_dir, "*.log")
        log_files = glob.glob(log_pattern)
        
        if not log_files:
            return None
        
        # Search for test accuracy in most recent log
        latest_log = max(log_files, key=os.path.getctime)
        
        try:
            with open(latest_log, 'r') as f:
                content = f.read()
                
            # Multiple patterns to extract test accuracy
            patterns = [
                r'Test Accuracy[:\s]+([0-9.]+)',
                r'Test loss[:\s]+[0-9.]+[:\s|]+Test Accuracy[:\s]+([0-9.]+)',
                r'test_acc[:\s]+([0-9.]+)',
                r'Test[:\s]+([0-9.]+)%'
            ]
            
            for pattern in patterns:
                matches = re.findall(pattern, content, re.IGNORECASE)
                if matches:
                    return float(matches[-1])  # Return last occurrence
            
            return None
            
        except Exception:
            return None
    
    def save_results(self):
        """Save optimization results to CSV and JSON."""
        if not self.results:
            return
            
        # Save to CSV
        df = pd.DataFrame(self.results)
        csv_path = os.path.join(self.config.results_dir, 'optimization_results.csv')
        df.to_csv(csv_path, index=False)
        
        # Save best result
        if self.best_result:
            best_path = os.path.join(self.config.results_dir, 'best_result.json')
            with open(best_path, 'w') as f:
                json.dump(self.best_result, f, indent=2)
        
        print(f"   💾 Results saved: {csv_path}")
    
    def prepare_base_models(self):
        """
        Prepare base models required for ft_1p optimization.
        This runs the self_supervised stage to create the required checkpoint.
        """
        print("🔧 PREPARING BASE MODELS")
        print("=" * 50)
        print("⏳ Running self_supervised stage to create base model...")
        
        try:
            # Create arguments for self_supervised mode
            args = argparse.Namespace(
                experiment_description=self.config.experiment_description,
                run_description=f"{self.config.run_description}_base_preparation",
                seed=self.config.seed,
                training_mode='self_supervised',
                selected_dataset=self.config.selected_dataset,
                data_path=self.config.data_path,
                logs_save_dir=self.config.logs_save_dir,
                device=self.config.device,
                home_path=self.config.home_path,
                enable_coft=True
            )
            
            # Run self_supervised training
            preparation_start_time = datetime.now()
            success = execute_training_mode(args, 'self_supervised', preparation_start_time)
            
            if success:
                preparation_duration = (datetime.now() - preparation_start_time).total_seconds()
                print(f"✅ Base model preparation completed in {preparation_duration:.1f} seconds")
                return True
            else:
                print("❌ Base model preparation failed!")
                return False
                
        except Exception as e:
            print(f"❌ Base model preparation failed: {str(e)}")
            return False
    
    def setup_base_model_for_trial(self, trial_number: int):
        """
        Setup base model checkpoint for a specific trial.
        Copies the prepared base model to the trial's expected location.
        """
        # Source: base model from preparation
        source_dir = os.path.join(
            self.config.logs_save_dir,
            self.config.experiment_description,
            f"{self.config.run_description}_base_preparation",
            f"self_supervised_seed_{self.config.seed}",
            "saved_models"
        )
        
        # Destination: where this trial expects to find the base model
        dest_dir = os.path.join(
            self.config.logs_save_dir,
            self.config.experiment_description,
            f"{self.config.run_description}_trial_{trial_number}",
            f"self_supervised_seed_{self.config.seed}",
            "saved_models"
        )
        
        # Create destination directory
        os.makedirs(dest_dir, exist_ok=True)
        
        # Copy checkpoint file
        source_file = os.path.join(source_dir, "ckp_last.pt")
        dest_file = os.path.join(dest_dir, "ckp_last.pt")
        
        if os.path.exists(source_file):
            shutil.copy2(source_file, dest_file)
            print(f"   📋 Copied base model for trial {trial_number}")
        else:
            print(f"   ⚠️ Warning: Base model not found at {source_file}")

    def run_optimization(self):
        """Run the optimization process."""
        n_trials = self.config.n_trials[self.config.optimization_mode]
        
        print(f"🚀 Starting CoFT Hyperparameter Optimization")
        print(f"   Mode: {self.config.optimization_mode}")
        print(f"   Trials: {n_trials}")
        print(f"   Dataset: {self.config.selected_dataset}")
        print(f"   Results: {self.config.results_dir}")
        print("=" * 80)
        
        # Prepare base models if needed for ft_1p mode
        if self.config.training_mode == 'ft_1p':
            print("🔍 Detected ft_1p mode - checking for required base models...")
            
            # Check if base model exists
            base_model_path = os.path.join(
                self.config.logs_save_dir,
                self.config.experiment_description,
                f"{self.config.run_description}_base_preparation",
                f"self_supervised_seed_{self.config.seed}",
                "saved_models",
                "ckp_last.pt"
            )
            
            if not os.path.exists(base_model_path):
                print("⚠️ Base model not found - preparing automatically...")
                if not self.prepare_base_models():
                    print("❌ Cannot proceed without base model!")
                    return None
            else:
                print("✅ Base model found - proceeding with optimization...")
        
        # Run optimization
        self.study.optimize(self.objective, n_trials=n_trials)
        
        # Final summary
        print("\\n" + "=" * 80)
        print("🏁 OPTIMIZATION COMPLETED")
        print("=" * 80)
        
        # Check if any trials succeeded
        try:
            if len(self.study.trials) > 0 and any(trial.state == optuna.trial.TrialState.COMPLETE for trial in self.study.trials):
                best_trial = self.study.best_trial
                print(f"🏆 Best Trial: {best_trial.number}")
                print(f"🎯 Best Accuracy: {best_trial.value:.4f}%")
                print("📋 Best Parameters:")
                for key, value in best_trial.params.items():
                    print(f"   {key}: {value}")
                print(f"📊 Total Trials: {len(self.results)}")
                print(f"💾 Results Directory: {self.config.results_dir}")
                return best_trial
            else:
                print("❌ No successful trials found!")
                print("💡 All trials were pruned due to errors.")
                print("🔍 Check the error messages above for debugging.")
                print(f"📊 Total Attempted Trials: {len(self.study.trials)}")
                print(f"💾 Results Directory: {self.config.results_dir}")
                return None
        except ValueError as e:
            print("❌ No successful trials found!")
            print("💡 All trials were pruned due to errors.")
            print("🔍 Check the error messages above for debugging.")
            print(f"📊 Total Attempted Trials: {len(self.study.trials)}")
            print(f"💾 Results Directory: {self.config.results_dir}")
            return None

# Reinitialize optimizer with the updated class
optimizer = CoFTHyperparameterOptimizer(config, param_manager)
print("🎯 Hyperparameter Optimizer initialized with base model preparation")
print(f"   Search strategy: {config.optimization_mode}")
print(f"   Ready to optimize {config.n_trials[config.optimization_mode]} trials")
print("   ✅ Auto base model preparation enabled for ft_1p mode")


[I 2025-06-22 19:37:55,470] A new study created in RDB with name: coft_optimization_HAR_diagnostic


🎯 Hyperparameter Optimizer initialized with base model preparation
   Search strategy: diagnostic
   Ready to optimize 5 trials
   ✅ Auto base model preparation enabled for ft_1p mode


In [5]:
# =============================================================================
# EXECUTE HYPERPARAMETER OPTIMIZATION
# =============================================================================

# Modify configuration if needed before running
print("🎯 Current Configuration:")
print(f"   Mode: {config.optimization_mode}")
print(f"   Dataset: {config.selected_dataset}")
print(f"   Trials: {config.n_trials[config.optimization_mode]}")
print(f"   Search Space: {config.search_spaces[config.optimization_mode]}")

# Option to change mode before running
# Uncomment and modify as needed:
config.optimization_mode = 'diagnostic'  # Change to 'diagnostic', 'quick', or 'full'
config.selected_dataset = 'HAR'     # Change dataset if needed

print("\\n🚨 IMPORTANT: This will modify CoFT source files temporarily during optimization")
print("🔄 Original files will be backed up and restored automatically")
print("⏱️ Expected duration:")
print(f"   - diagnostic mode: ~15-30 minutes ({config.n_trials['diagnostic']} trials)")
print(f"   - quick mode: ~1-2 hours ({config.n_trials['quick']} trials)")
print(f"   - full mode: ~4-8 hours ({config.n_trials['full']} trials)")

# Uncomment the line below to start optimization
print("\\n💡 To start optimization, uncomment and run: best_trial = optimizer.run_optimization()")

# 🚀 NOTEBOOK EXECUTION SECTION - Choose your option:

# Option 1: Test parameter updates only (recommended first)
print("\\n🧪 TESTING PARAMETER MANAGER (Fixed Version)")
print("=" * 60)
param_manager.backup_files()
param_manager.update_coft_loss_params(0.005, 0.1) 
param_manager.update_ensemble_method('simple_average')
verification = param_manager.verify_parameters(0.005, 0.1, 'simple_average')
param_manager.restore_files()
print(f"✅ Parameter manager test: {verification}/3 verified")

# Option 2: Run single test trial (quick validation)
print("\\n🧪 QUICK VALIDATION - Single trial test")
print("This will run 1 trial to validate fixes")
# Uncomment to run: config.n_trials['diagnostic'] = 1; best_trial = optimizer.run_optimization()

# Option 3: Run full diagnostic optimization (5 trials) 
print("\\n🚀 FULL DIAGNOSTIC - 5 trials with all fixes")
print("This includes gradient clipping, ensemble fixes, better verification")
# Uncomment to run: best_trial = optimizer.run_optimization()


🎯 Current Configuration:
   Mode: diagnostic
   Dataset: HAR
   Trials: 5
   Search Space: {'lambda_cotraining': [0.001, 0.005, 0.01], 'lambda_consistency': [0.1], 'ensemble_method': ['temporal_only', 'simple_average']}
\n🚨 IMPORTANT: This will modify CoFT source files temporarily during optimization
🔄 Original files will be backed up and restored automatically
⏱️ Expected duration:
   - diagnostic mode: ~15-30 minutes (5 trials)
   - quick mode: ~1-2 hours (20 trials)
   - full mode: ~4-8 hours (100 trials)
\n💡 To start optimization, uncomment and run: best_trial = optimizer.run_optimization()
\n🧪 TESTING PARAMETER MANAGER (Fixed Version)
   📝 Backed up: models/coft_loss.py
   📝 Backed up: trainer/trainer_coft.py
   ✅ Updated λ_cotraining=0.005, λ_consistency=0.1
   ✅ Updated ensemble method: simple_average
   ✅ lambda_cotraining = 0.005 verified
   ✅ lambda_consistency = 0.1 verified
   ✅ Ensemble method: simple_average verified
   🔄 Restored: models/coft_loss.py
   🔄 Restored: train

In [6]:
# =============================================================================
# RESULTS VISUALIZATION AND ANALYSIS
# =============================================================================

def create_optimization_dashboard(optimizer: CoFTHyperparameterOptimizer):
    """Create interactive dashboard for optimization results."""
    
    if not optimizer.results:
        print("⚠️ No results available yet. Run optimization first.")
        return
    
    df = pd.DataFrame(optimizer.results)
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Optimization Progress',
            'Parameter vs Accuracy',
            'Parameter Distribution', 
            'Trial Duration Analysis'
        ],
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # 1. Optimization Progress
    fig.add_trace(
        go.Scatter(
            x=df['trial'],
            y=df['test_accuracy'],
            mode='lines+markers',
            name='Test Accuracy',
            line=dict(color='blue'),
            hovertemplate='Trial: %{x}<br>Accuracy: %{y:.4f}%<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Add best trial marker
    if optimizer.best_result:
        fig.add_trace(
            go.Scatter(
                x=[optimizer.best_result['trial']],
                y=[optimizer.best_result['test_accuracy']],
                mode='markers',
                name='Best Trial',
                marker=dict(color='red', size=15, symbol='star'),
                hovertemplate=f'Best Trial: {optimizer.best_result["trial"]}<br>Best Accuracy: {optimizer.best_result["test_accuracy"]:.4f}%<extra></extra>'
            ),
            row=1, col=1
        )
    
    # 2. Parameter vs Accuracy scatter
    fig.add_trace(
        go.Scatter(
            x=df['lambda_cotraining'],
            y=df['test_accuracy'],
            mode='markers',
            name='λ_cotraining vs Accuracy',
            marker=dict(
                color=df['test_accuracy'],
                colorscale='Viridis',
                size=8,
                colorbar=dict(title="Accuracy %")
            ),
            text=df['ensemble_method'],
            hovertemplate='λ_cotraining: %{x}<br>Accuracy: %{y:.4f}%<br>Ensemble: %{text}<extra></extra>'
        ),
        row=1, col=2
    )
    
    # 3. Parameter distribution
    fig.add_trace(
        go.Histogram(
            x=df['lambda_cotraining'],
            name='λ_cotraining Distribution',
            nbinsx=20,
            marker_color='lightblue'
        ),
        row=2, col=1
    )
    
    # 4. Trial duration
    fig.add_trace(
        go.Scatter(
            x=df['trial'],
            y=df['duration_seconds'] / 60,  # Convert to minutes
            mode='lines+markers',
            name='Duration (minutes)',
            line=dict(color='orange'),
            hovertemplate='Trial: %{x}<br>Duration: %{y:.1f} min<extra></extra>'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text="CoFT Hyperparameter Optimization Dashboard",
        showlegend=True,
        height=800,
        template='plotly_white'
    )
    
    # Update axes labels
    fig.update_xaxes(title_text="Trial Number", row=1, col=1)
    fig.update_yaxes(title_text="Test Accuracy (%)", row=1, col=1)
    
    fig.update_xaxes(title_text="λ_cotraining", row=1, col=2)
    fig.update_yaxes(title_text="Test Accuracy (%)", row=1, col=2)
    
    fig.update_xaxes(title_text="λ_cotraining", row=2, col=1)
    fig.update_yaxes(title_text="Frequency", row=2, col=1)
    
    fig.update_xaxes(title_text="Trial Number", row=2, col=2)
    fig.update_yaxes(title_text="Duration (minutes)", row=2, col=2)
    
    fig.show()
    
    # Print summary statistics
    print("\\n📊 OPTIMIZATION SUMMARY")
    print("=" * 50)
    print(f"Total Trials: {len(df)}")
    print(f"Best Accuracy: {df['test_accuracy'].max():.4f}%")
    print(f"Mean Accuracy: {df['test_accuracy'].mean():.4f}%")
    print(f"Std Accuracy: {df['test_accuracy'].std():.4f}%")
    print(f"Average Duration: {df['duration_seconds'].mean() / 60:.1f} minutes")
    
    if optimizer.best_result:
        print("\\n🏆 BEST CONFIGURATION")
        print("=" * 50)
        for key, value in optimizer.best_result.items():
            if key not in ['timestamp']:
                print(f"{key}: {value}")

def analyze_parameter_sensitivity(optimizer: CoFTHyperparameterOptimizer):
    """Analyze parameter sensitivity and correlations."""
    
    if not optimizer.results:
        print("⚠️ No results available yet. Run optimization first.")
        return
    
    df = pd.DataFrame(optimizer.results)
    
    # Correlation analysis
    numeric_cols = ['lambda_cotraining', 'lambda_consistency', 'test_accuracy', 'duration_seconds']
    correlation_matrix = df[numeric_cols].corr()
    
    # Create correlation heatmap
    fig = px.imshow(
        correlation_matrix,
        text_auto=True,
        aspect="auto",
        title="Parameter Correlation Matrix",
        color_continuous_scale='RdBu'
    )
    fig.show()
    
    # Parameter sensitivity analysis
    print("\\n🔍 PARAMETER SENSITIVITY ANALYSIS")
    print("=" * 50)
    
    # Lambda cotraining analysis
    lambda_ct_corr = df['lambda_cotraining'].corr(df['test_accuracy'])
    print(f"λ_cotraining correlation with accuracy: {lambda_ct_corr:.4f}")
    
    # Lambda consistency analysis
    lambda_cs_corr = df['lambda_consistency'].corr(df['test_accuracy'])
    print(f"λ_consistency correlation with accuracy: {lambda_cs_corr:.4f}")
    
    # Ensemble method analysis
    ensemble_analysis = df.groupby('ensemble_method')['test_accuracy'].agg(['mean', 'std', 'count'])
    print(f"\\nEnsemble method performance:")
    print(ensemble_analysis)
    
    # Top configurations
    print(f"\\n🏆 TOP 5 CONFIGURATIONS:")
    print("=" * 50)
    top_5 = df.nlargest(5, 'test_accuracy')[['trial', 'lambda_cotraining', 'lambda_consistency', 'ensemble_method', 'test_accuracy']]
    print(top_5.to_string(index=False))

def export_best_configuration(optimizer: CoFTHyperparameterOptimizer):
    """Export best configuration for production use."""
    
    if not optimizer.best_result:
        print("⚠️ No best result available yet. Run optimization first.")
        return
    
    best_config = {
        'lambda_cotraining': optimizer.best_result['lambda_cotraining'],
        'lambda_consistency': optimizer.best_result['lambda_consistency'],
        'ensemble_method': optimizer.best_result['ensemble_method'],
        'test_accuracy': optimizer.best_result['test_accuracy'],
        'optimization_metadata': {
            'trial': optimizer.best_result['trial'],
            'dataset': config.selected_dataset,
            'optimization_mode': config.optimization_mode,
            'timestamp': optimizer.best_result['timestamp']
        }
    }
    
    # Save configuration
    config_path = os.path.join(config.results_dir, 'best_configuration.json')
    with open(config_path, 'w') as f:
        json.dump(best_config, f, indent=2)
    
    print("🎯 BEST CONFIGURATION EXPORTED")
    print("=" * 50)
    print(f"Configuration saved to: {config_path}")
    print("\\nTo apply this configuration:")
    print("1. Update models/coft_loss.py:")
    print(f"   self.lambda_cotraining = {best_config['lambda_cotraining']}")
    print(f"   self.lambda_consistency = {best_config['lambda_consistency']}")
    print(f"2. Set ensemble method to: {best_config['ensemble_method']}")
    print(f"3. Expected accuracy: {best_config['test_accuracy']:.4f}%")
    
    return best_config

# Load existing results if available
def load_previous_results():
    """Load results from previous optimization runs."""
    results_file = os.path.join(config.results_dir, 'optimization_results.csv')
    
    if os.path.exists(results_file):
        df = pd.read_csv(results_file)
        optimizer.results = df.to_dict('records')
        
        # Find best result
        best_idx = df['test_accuracy'].idxmax()
        optimizer.best_result = df.loc[best_idx].to_dict()
        
        print(f"✅ Loaded {len(optimizer.results)} previous results")
        print(f"🏆 Best previous result: {optimizer.best_result['test_accuracy']:.4f}%")
    else:
        print("ℹ️ No previous results found")

# Example usage (uncomment after running optimization):
print("📊 Visualization functions ready:")
print("   - create_optimization_dashboard(optimizer)")
print("   - analyze_parameter_sensitivity(optimizer)")  
print("   - export_best_configuration(optimizer)")
print("   - load_previous_results()")

# Load any existing results
load_previous_results()


📊 Visualization functions ready:
   - create_optimization_dashboard(optimizer)
   - analyze_parameter_sensitivity(optimizer)
   - export_best_configuration(optimizer)
   - load_previous_results()
ℹ️ No previous results found
